# 03 — NumPy for ML Algorithms

**What you'll know after this notebook:**
- Implement the ML building blocks that appear in every exam notebook
- Sigmoid, feature normalization, MSE cost, gradient descent, one-hot encoding
- PCA eigendecomposition pipeline
- K-means distance computations
- Backpropagation helper functions

**Time estimate:** 60–90 minutes

This notebook builds directly on [02_numpy_core.ipynb](02_numpy_core.ipynb). Each section gives you a pattern and then asks you to implement a variant.

In [2]:
import numpy as np
np.random.seed(42)

## 1. Sigmoid and Its Gradient

Used in: Logistic Regression, Neural Networks (every hidden and output layer)

In [3]:
def sigmoid(z):
    """σ(z) = 1 / (1 + e^{-z}). Works on scalars, 1D, 2D arrays."""
    return 1.0 / (1.0 + np.exp(-z))

# Properties to remember:
# sigmoid(0)   = 0.5
# sigmoid(+∞) ≈ 1
# sigmoid(-∞) ≈ 0
# Output is always in (0, 1) → usable as probability

test_values = np.array([-10, -2, -1, 0, 1, 2, 10])
print(np.round(sigmoid(test_values), 4))

[0.     0.1192 0.2689 0.5    0.7311 0.8808 1.    ]


In [4]:
def sigmoid_gradient(z):
    """Derivative of sigmoid: σ'(z) = σ(z) * (1 - σ(z))"""
    s = sigmoid(z)
    return s * (1 - s)

# Maximum gradient at z=0: sigmoid_gradient(0) = 0.5 * 0.5 = 0.25
# Vanishes at extremes (gradient vanishing problem in deep networks)

z = np.array([-3, -1, 0, 1, 3])
print("sigmoid:          ", np.round(sigmoid(z), 4))
print("sigmoid_gradient: ", np.round(sigmoid_gradient(z), 4))

sigmoid:           [0.0474 0.2689 0.5    0.7311 0.9526]
sigmoid_gradient:  [0.0452 0.1966 0.25   0.1966 0.0452]


## 2. Feature Normalization

Used in: EVERY regression and neural network notebook. Without this, gradient descent diverges.

In [5]:
def feature_normalize(X):
    """
    Z-score normalization: X_norm = (X - mean) / std
    Returns X_norm, mu, sigma so you can normalize test data the same way.
    """
    mu = X.mean(axis=0)     # shape (n,) — one mean per feature
    sigma = X.std(axis=0)   # shape (n,) — one std per feature
    X_norm = (X - mu) / sigma    # broadcast: (m,n) - (n,) / (n,)
    return X_norm, mu, sigma

# Test
X = np.array([[1.0,  100.0, 0.01],
              [2.0,  200.0, 0.02],
              [3.0,  300.0, 0.03],
              [4.0,  400.0, 0.04]])

X_norm, mu, sigma = feature_normalize(X)
print("X_norm:")
print(X_norm)
print("Means (should be 0):", X_norm.mean(axis=0).round(10))
print("Stds  (should be 1):", X_norm.std(axis=0).round(10))

X_norm:
[[-1.34164079 -1.34164079 -1.34164079]
 [-0.4472136  -0.4472136  -0.4472136 ]
 [ 0.4472136   0.4472136   0.4472136 ]
 [ 1.34164079  1.34164079  1.34164079]]
Means (should be 0): [ 0.  0. -0.]
Stds  (should be 1): [1. 1. 1.]


In [6]:
# CRITICAL: Use the TRAINING set's mu and sigma on test data too
# Never normalize test data with its own statistics

X_train = np.random.randn(80, 3) * np.array([1, 100, 0.01])
X_test  = np.random.randn(20, 3) * np.array([1, 100, 0.01])

X_train_norm, mu, sigma = feature_normalize(X_train)
X_test_norm = (X_test - mu) / sigma    # use TRAINING mu and sigma!

## 3. Cost Functions

### 3a. MSE (Linear Regression)

In [7]:
def compute_cost_linear(X, y, W):
    """
    J(W) = (1/2m) * ||X @ W - y||^2
    X: (m, n+1), y: (m,), W: (n+1,)
    """
    m = len(y)
    error = X @ W - y           # (m,)
    return (1 / (2 * m)) * (error @ error)   # error^T @ error = sum of squares

# Test with known solution: W_true makes cost ≈ 0
np.random.seed(0)
m, n = 30, 2
X = np.hstack([np.ones((m, 1)), np.random.randn(m, n)])
W_true = np.array([1.0, -2.0, 3.0])
y = X @ W_true  # no noise

print("Cost at W_true:", compute_cost_linear(X, y, W_true))  # ~0
print("Cost at W=0:   ", compute_cost_linear(X, y, np.zeros(3)))

Cost at W_true: 0.0
Cost at W=0:    7.681439898128864


### 3b. Binary Cross-Entropy (Logistic Regression)

In [8]:
def compute_cost_logistic(W, X, y):
    """
    J(W) = -(1/m) * [y^T log(h) + (1-y)^T log(1-h)]
    where h = sigmoid(X @ W)
    """
    m = len(y)
    h = sigmoid(X @ W)                          # predictions in (0,1)
    epsilon = 1e-8                               # numerical stability
    cost = -(1/m) * (y @ np.log(h + epsilon) + 
                     (1 - y) @ np.log(1 - h + epsilon))
    return cost

# Note: W comes FIRST — matches scipy.optimize convention in the practice notebooks

m, n = 50, 3
X = np.hstack([np.ones((m, 1)), np.random.randn(m, n)])
y = (np.random.rand(m) > 0.5).astype(float)
W = np.zeros(n + 1)

print("Initial cost:", compute_cost_logistic(W, X, y))

Initial cost: 0.6931471605599454


### 3c. Regularized Cost

In [9]:
def compute_cost_reg(W, X, y, lam=1.0):
    """
    Regularized logistic regression cost:
    J_reg = J + (lambda / 2m) * sum(W[1:]^2)
    Note: do NOT regularize the bias term W[0]
    """
    m = len(y)
    cost = compute_cost_logistic(W, X, y)
    reg_term = (lam / (2 * m)) * (W[1:] @ W[1:])  # skip W[0] (bias)
    return cost + reg_term

W = np.random.randn(n + 1)
print("Unregularized:", compute_cost_logistic(W, X, y))
print("Regularized (λ=1):", compute_cost_reg(W, X, y, lam=1.0))
print("Regularized (λ=10):", compute_cost_reg(W, X, y, lam=10.0))

Unregularized: 1.2625951217980569
Regularized (λ=1): 1.3156194527810534
Regularized (λ=10): 1.792838431628022


## 4. Gradient Descent

In [ ]:
def gradient_descent_linear(X, y, W, lr=0.01, epochs=100):
    """
    Gradient of MSE w.r.t. W: (1/m) * X.T @ (X @ W - y)
    """
    m = len(y)
    cost_history = []
    
    for _ in range(epochs):
        error = X @ W - y                            # (m,)
        gradient = (1 / m) * (X.T @ error)           # (n+1,)
        W = W - lr * gradient
        cost_history.append(compute_cost_linear(X, y, W))
    
    return W, cost_history

W_init = np.zeros(n + 1)
W_opt, costs = gradient_descent_linear(X, y, W_init, lr=0.1, epochs=500)
print(f"Initial cost: {costs[0]:.4f}")
print(f"Final cost:   {costs[-1]:.4f}")
print(f"Cost reduced: {(costs[0]-costs[-1])/costs[0]*100:.1f}%")

In [10]:
def gradient_descent_logistic(X, y, W, lr=0.01, epochs=100, lam=0.0):
    """
    Logistic regression gradient: (1/m) * X.T @ (sigmoid(X @ W) - y)
    With optional L2 regularization on W[1:]
    """
    m = len(y)
    cost_history = []
    
    for _ in range(epochs):
        h = sigmoid(X @ W)                          # (m,)
        error = h - y                               # (m,)
        gradient = (1 / m) * (X.T @ error)          # (n+1,)
        
        if lam > 0:
            reg = (lam / m) * W.copy()
            reg[0] = 0                              # don't regularize bias
            gradient += reg
        
        W = W - lr * gradient
        cost_history.append(compute_cost_reg(W, X, y, lam))
    
    return W, cost_history

W_init = np.zeros(n + 1)
W_log, log_costs = gradient_descent_logistic(X, y, W_init, lr=0.5, epochs=300)
predictions = (sigmoid(X @ W_log) >= 0.5).astype(int)
accuracy = np.mean(predictions == y.astype(int))
print(f"Training accuracy: {accuracy:.2%}")

Training accuracy: 56.00%


## 5. One-Hot Encoding

In [11]:
def to_one_hot(y, num_classes):
    """
    Convert integer label array to one-hot matrix.
    y: shape (m,) with integer values in [0, num_classes)
    Returns: shape (m, num_classes)
    """
    m = len(y)
    result = np.zeros((m, num_classes))
    result[np.arange(m), y] = 1   # row i gets a 1 at column y[i]
    return result

y = np.array([3, 0, 7, 2, 9, 5])
Y = to_one_hot(y, 10)
print("Shape:", Y.shape)     # (6, 10)
print("Row 0:", Y[0])         # [0 0 0 1 0 0 0 0 0 0] — label 3
print("Row 1:", Y[1])         # [1 0 0 0 0 0 0 0 0 0] — label 0

# Reverse: get integer labels from one-hot
y_recovered = Y.argmax(axis=1)
print("Recovered labels:", y_recovered)

Shape: (6, 10)
Row 0: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
Row 1: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Recovered labels: [3 0 7 2 9 5]


## 6. Neural Network Forward Pass

In [ ]:
def forward_pass(X, W1, W2):
    """
    Two-layer neural network forward pass.
    X:  (m, input_size)         — input (with bias added inside)
    W1: (input_size+1, hidden)  — input-to-hidden weights (includes bias row)
    W2: (hidden+1, output)      — hidden-to-output weights (includes bias row)
    Returns: output probabilities (m, output_size)
    """
    m = X.shape[0]
    
    # Layer 1: input → hidden
    a0 = np.hstack([np.ones((m, 1)), X])    # add bias: (m, input_size+1)
    z1 = a0 @ W1                             # (m, hidden)
    a1 = sigmoid(z1)                         # (m, hidden)
    
    # Layer 2: hidden → output
    a1_bias = np.hstack([np.ones((m, 1)), a1])  # add bias: (m, hidden+1)
    z2 = a1_bias @ W2                            # (m, output)
    a2 = sigmoid(z2)                             # (m, output) — final probabilities
    
    return a2

# Example: 784 input, 25 hidden, 10 output (MNIST scale)
m = 10
input_size, hidden_size, output_size = 784, 25, 10

X = np.random.randn(m, input_size)
W1 = np.random.randn(input_size + 1, hidden_size) * 0.01
W2 = np.random.randn(hidden_size + 1, output_size) * 0.01

output = forward_pass(X, W1, W2)
print("Output shape:", output.shape)    # (10, 10)
print("Row sums (each should be ~# classes, since sigmoid not softmax):", output.sum(axis=1).round(2))
print("Predictions:", output.argmax(axis=1))

## 7. Backpropagation Building Blocks

In [ ]:
def backprop_step(X, y_onehot, W1, W2, lr=0.1):
    """
    One gradient descent step using backpropagation.
    Implements the chain rule for a 2-layer sigmoid network.
    """
    m = X.shape[0]
    
    # === FORWARD PASS ===
    a0 = np.hstack([np.ones((m, 1)), X])        # (m, input+1)
    z1 = a0 @ W1                                 # (m, hidden)
    a1 = sigmoid(z1)                             # (m, hidden)
    a1_b = np.hstack([np.ones((m, 1)), a1])      # (m, hidden+1)
    z2 = a1_b @ W2                               # (m, output)
    a2 = sigmoid(z2)                             # (m, output)
    
    # === BACKWARD PASS ===
    # Output layer error
    delta2 = a2 - y_onehot                       # (m, output)
    
    # Hidden layer error (don't propagate through bias node)
    delta1 = (delta2 @ W2.T)[:, 1:] * sigmoid_gradient(z1)  # (m, hidden)
    #                            ^^^^ skip bias column
    
    # Gradients
    grad_W2 = (1/m) * a1_b.T @ delta2            # (hidden+1, output)
    grad_W1 = (1/m) * a0.T @ delta1              # (input+1, hidden)
    
    # Update
    W1_new = W1 - lr * grad_W1
    W2_new = W2 - lr * grad_W2
    
    # Loss (for monitoring)
    loss = -np.mean(y_onehot * np.log(a2 + 1e-8))
    
    return W1_new, W2_new, loss

# Quick smoke test (not training to convergence here)
np.random.seed(0)
m = 20
X_nn = np.random.randn(m, 4)     # 4 features
y_nn = np.random.randint(0, 3, m)
Y_nn = to_one_hot(y_nn, 3)

W1_nn = np.random.randn(5, 5) * 0.1   # (input+1, hidden)
W2_nn = np.random.randn(6, 3) * 0.1   # (hidden+1, output)

losses = []
for epoch in range(200):
    W1_nn, W2_nn, loss = backprop_step(X_nn, Y_nn, W1_nn, W2_nn, lr=0.5)
    losses.append(loss)

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")

## 8. K-Means Algorithm

In [ ]:
def find_closest_centroids(X, centroids):
    """
    Assign each sample to the nearest centroid.
    X:         (m, n)
    centroids: (K, n)
    Returns:   (m,) array of centroid indices
    """
    m = X.shape[0]
    K = centroids.shape[0]
    
    # Compute distance from each sample to each centroid
    # distances[i, k] = ||X[i] - centroids[k]||^2
    distances = np.zeros((m, K))
    for k in range(K):
        diff = X - centroids[k]              # broadcast: (m,n) - (n,)
        distances[:, k] = (diff**2).sum(axis=1)   # squared Euclidean distance
    
    return distances.argmin(axis=1)          # (m,) — index of nearest centroid


def compute_centroids(X, assignments, K):
    """
    Recompute centroids as the mean of assigned samples.
    """
    n = X.shape[1]
    centroids = np.zeros((K, n))
    for k in range(K):
        cluster_k = X[assignments == k]        # select rows assigned to cluster k
        if len(cluster_k) > 0:
            centroids[k] = cluster_k.mean(axis=0)
    return centroids


def run_kmeans(X, K, max_iters=100):
    """Full K-means algorithm."""
    # Initialize centroids randomly from data points
    idx = np.random.choice(len(X), K, replace=False)
    centroids = X[idx].copy()
    
    for _ in range(max_iters):
        assignments = find_closest_centroids(X, centroids)
        new_centroids = compute_centroids(X, assignments, K)
        if np.allclose(centroids, new_centroids):  # converged
            break
        centroids = new_centroids
    
    return assignments, centroids

# Test on synthetic data
np.random.seed(0)
cluster1 = np.random.randn(50, 2) + np.array([0, 0])
cluster2 = np.random.randn(50, 2) + np.array([5, 5])
cluster3 = np.random.randn(50, 2) + np.array([0, 10])
X_kmeans = np.vstack([cluster1, cluster2, cluster3])

assignments, centroids = run_kmeans(X_kmeans, K=3)
print("Centroid 0:", centroids[0].round(2))  # should be near [0, 0]
print("Centroid 1:", centroids[1].round(2))  # should be near [5, 5]
print("Centroid 2:", centroids[2].round(2))  # should be near [0, 10]

## 9. PCA Pipeline

In [ ]:
def pca(X, num_components):
    """
    Principal Component Analysis.
    Returns reduced data (m, num_components) and explained variance ratio.
    """
    m, n = X.shape
    
    # Step 1: Normalize
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    X_norm = (X - mu) / sigma
    
    # Step 2: Covariance matrix
    Sigma = (1 / m) * (X_norm.T @ X_norm)   # shape (n, n)
    
    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eig(Sigma)
    
    # Step 4: Sort by eigenvalue descending
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order].real      # .real to discard tiny imaginary parts
    eigenvectors = eigenvectors[:, order].real  # columns = principal components
    
    # Step 5: Select top k components
    U = eigenvectors[:, :num_components]        # (n, k)
    
    # Step 6: Project
    X_reduced = X_norm @ U                      # (m, k)
    
    # Explained variance ratio
    explained_variance = eigenvalues[:num_components].sum() / eigenvalues.sum()
    
    return X_reduced, U, explained_variance

# Test: reduce 5D data to 2D
np.random.seed(0)
X_pca = np.random.randn(100, 5)
X_pca[:, 2] = X_pca[:, 0] + 0.1 * np.random.randn(100)  # correlated features

X_2d, U, var_ratio = pca(X_pca, num_components=2)
print("Input shape: ", X_pca.shape)   # (100, 5)
print("Reduced shape:", X_2d.shape)   # (100, 2)
print(f"Explained variance: {var_ratio:.2%}")

## 10. Polynomial Feature Expansion

In [ ]:
def map_feature(X1, X2, degree=6):
    """
    Generate polynomial features from two input features.
    Returns: matrix with columns [1, x1, x2, x1^2, x1*x2, x2^2, ...] up to given degree.
    Used in regularized logistic regression for non-linear decision boundaries.
    """
    m = len(X1)
    cols = [np.ones(m)]  # bias column
    
    for i in range(1, degree + 1):
        for j in range(i + 1):
            cols.append((X1 ** (i - j)) * (X2 ** j))
    
    return np.column_stack(cols)

# Test
x1 = np.array([1.0, 2.0, 3.0])
x2 = np.array([1.0, 1.0, 2.0])
X_poly = map_feature(x1, x2, degree=2)

print("Degree 2 features shape:", X_poly.shape)  # (3, 6)
print("Columns: [1, x1, x2, x1^2, x1*x2, x2^2]")
print(X_poly)

---
## Exercises

### Exercise 1: Sigmoid Sanity Checks
Verify: `sigmoid(0) == 0.5`, `sigmoid(z) + sigmoid(-z) == 1`, `sigmoid_gradient(0) == 0.25`

In [ ]:
# YOUR CODE HERE: run the three checks above and print True/False for each
z = np.array([-5, -2, 0, 2, 5])

In [ ]:
# SOLUTION
z = np.array([-5, -2, 0, 2, 5])
print("sigmoid(0) == 0.5:", np.isclose(sigmoid(0), 0.5))
print("sigmoid(z)+sigmoid(-z)==1:", np.allclose(sigmoid(z) + sigmoid(-z), 1))
print("sigmoid_gradient(0) == 0.25:", np.isclose(sigmoid_gradient(0), 0.25))

### Exercise 2: Implement `predict_logistic`
Given weights `W` and data `X`, return binary predictions (0 or 1) using a 0.5 threshold.

In [ ]:
def predict_logistic(W, X):
    # YOUR CODE HERE
    pass

# Test
W_test = np.array([0.0, 2.0, -1.0])  # W[0]=bias, W[1]=pos feature, W[2]=neg feature
X_test = np.array([[1, 3, 0],   # strong positive signal → should predict 1
                   [1, 0, 5],   # strong negative signal → should predict 0
                   [1, 1, 1]])  # neutral

print(predict_logistic(W_test, X_test))  # Expected: [1, 0, ...]

In [ ]:
# SOLUTION
def predict_logistic(W, X):
    return (sigmoid(X @ W) >= 0.5).astype(int)

W_test = np.array([0.0, 2.0, -1.0])
X_test = np.array([[1, 3, 0], [1, 0, 5], [1, 1, 1]])
print(predict_logistic(W_test, X_test))

### Exercise 3: One-vs-All Classifier
Implement `one_vs_all_predict(W_all, X)` that takes a weight matrix `W_all` of shape `(K, n+1)` (one weight vector per class) and returns the predicted class (0 to K-1) for each sample.

In [ ]:
def one_vs_all_predict(W_all, X):
    """
    W_all: (K, n+1) — K classifiers
    X:     (m, n+1) — m samples with bias
    Returns: (m,) array of predicted class indices
    """
    # YOUR CODE HERE
    # Hint: compute sigmoid(X @ W_all.T) to get (m, K) probability matrix
    # then take argmax across classes
    pass

# Test
K, m, n = 3, 10, 4
W_all = np.random.randn(K, n + 1)
X_ova = np.hstack([np.ones((m, 1)), np.random.randn(m, n)])
preds = one_vs_all_predict(W_all, X_ova)
print("Predictions shape:", preds.shape)  # (10,)
print("Predictions:", preds)              # integers in [0, 2]

In [ ]:
# SOLUTION
def one_vs_all_predict(W_all, X):
    probs = sigmoid(X @ W_all.T)     # (m, K)
    return probs.argmax(axis=1)      # (m,) — class with highest probability

preds = one_vs_all_predict(W_all, X_ova)
print(preds)

### Exercise 4: Implement `compute_centroids` without a loop
The version above uses a Python `for` loop over K. Implement it with pure NumPy (use boolean indexing for each cluster — or try `np.einsum` / smart broadcasting).

In [ ]:
def compute_centroids_fast(X, assignments, K):
    """
    Same as compute_centroids but try with a list comprehension instead of a for loop.
    X:           (m, n)
    assignments: (m,) integers in [0, K)
    Returns:     (K, n)
    """
    # YOUR CODE HERE
    pass

# Test: should match the loop-based version
X_test = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
asgn = np.array([0, 0, 1, 1])
print(compute_centroids_fast(X_test, asgn, K=2))
# Expected: [[2. 3.], [6. 7.]]

In [ ]:
# SOLUTION
def compute_centroids_fast(X, assignments, K):
    return np.array([X[assignments == k].mean(axis=0) for k in range(K)])

print(compute_centroids_fast(X_test, asgn, K=2))

### Exercise 5: Gradient Descent with Cost History
Implement full logistic regression training and verify the cost decreases monotonically.

In [ ]:
# Generate linearly separable data
np.random.seed(42)
m = 100
X_pos = np.random.randn(m // 2, 2) + np.array([2, 2])
X_neg = np.random.randn(m // 2, 2) + np.array([-2, -2])
X_data = np.vstack([X_pos, X_neg])
y_data = np.array([1] * (m // 2) + [0] * (m // 2), dtype=float)

# Add bias column
X_data = np.hstack([np.ones((m, 1)), X_data])

# Train using gradient_descent_logistic from above
W_final, cost_hist = gradient_descent_logistic(X_data, y_data, np.zeros(3), lr=0.5, epochs=200)

# YOUR CODE HERE: verify cost is decreasing
# 1. Print initial and final cost
# 2. Check that cost_hist[-1] < cost_hist[0]
# 3. Compute training accuracy

In [ ]:
# SOLUTION
W_final, cost_hist = gradient_descent_logistic(X_data, y_data, np.zeros(3), lr=0.5, epochs=200)

print(f"Initial cost: {cost_hist[0]:.4f}")
print(f"Final cost:   {cost_hist[-1]:.4f}")
print(f"Cost decreased: {cost_hist[-1] < cost_hist[0]}")

preds = predict_logistic(W_final, X_data)
print(f"Training accuracy: {np.mean(preds == y_data.astype(int)):.2%}")

### Exercise 6–10: Trace the Shapes
For each forward pass below, write down the shape at each step before running.

In [ ]:
# Ex 6: Neural network forward pass shapes
m, inp, hid, out = 32, 784, 25, 10
X_ = np.ones((m, inp))
W1_ = np.ones((inp + 1, hid))
W2_ = np.ones((hid + 1, out))

a0_ = np.hstack([np.ones((m, 1)), X_])   # shape? ___
z1_ = a0_ @ W1_                           # shape? ___
a1_ = sigmoid(z1_)                        # shape? ___
a1b_ = np.hstack([np.ones((m, 1)), a1_]) # shape? ___
z2_ = a1b_ @ W2_                          # shape? ___
a2_ = sigmoid(z2_)                        # shape? ___

print(a0_.shape, z1_.shape, a1_.shape, a1b_.shape, z2_.shape, a2_.shape)

In [ ]:
# Ex 7: PCA shapes
X_p = np.random.randn(200, 10)
Sigma_p = (1/200) * X_p.T @ X_p     # shape? ___
_, V = np.linalg.eig(Sigma_p)        # V shape? ___
U_k = V[:, :3]                        # shape? ___
X_r = X_p @ U_k                       # shape? ___
print(Sigma_p.shape, V.shape, U_k.shape, X_r.shape)

In [ ]:
# Ex 8: Gradient of logistic regression
m_, n_ = 100, 5
X_g = np.random.randn(m_, n_ + 1)     # shape (100, 6)
W_g = np.random.randn(n_ + 1)         # shape (6,)
y_g = np.random.rand(m_)              # shape (100,)

h_g = sigmoid(X_g @ W_g)              # shape? ___
err_g = h_g - y_g                      # shape? ___
grad_g = (1/m_) * (X_g.T @ err_g)     # shape? ___
W_new = W_g - 0.01 * grad_g           # shape? ___

print(h_g.shape, err_g.shape, grad_g.shape, W_new.shape)

In [ ]:
# Ex 9: K-means distance matrix (no loop version)
# Compute squared distance from each of m samples to each of K centroids
m_, n_, K_ = 50, 3, 4
X_km = np.random.randn(m_, n_)      # (50, 3)
C_km = np.random.randn(K_, n_)      # (4, 3)

# Vectorized: expand dims to broadcast (m,1,n) - (1,K,n) = (m,K,n), then sum over n
diff = X_km[:, np.newaxis, :] - C_km[np.newaxis, :, :]   # shape? ___
sq_dist = (diff**2).sum(axis=2)    # shape? ___
assignments = sq_dist.argmin(axis=1)  # shape? ___

print(diff.shape, sq_dist.shape, assignments.shape)

In [ ]:
# Ex 10: Backprop — output layer delta shape check
m_, hid_, out_ = 20, 25, 10
a2_ = np.random.rand(m_, out_)       # (20, 10)
y_oh = to_one_hot(np.random.randint(0, 10, m_), 10)  # (20, 10)
W2_ = np.random.randn(hid_ + 1, out_)  # (26, 10)
z1_ = np.random.randn(m_, hid_)         # (20, 25)

delta2 = a2_ - y_oh                     # shape? ___
back = delta2 @ W2_.T                   # shape? ___
delta1 = back[:, 1:] * sigmoid_gradient(z1_)  # shape? ___

print(delta2.shape, back.shape, delta1.shape)

---
## Summary — Exam Cheat Sheet

```python
# Sigmoid
sigmoid(z) = 1 / (1 + np.exp(-z))
sigmoid_gradient(z) = sigmoid(z) * (1 - sigmoid(z))

# Feature normalize
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)

# Linear regression cost
error = X @ W - y
J = (1/(2*m)) * (error @ error)

# Logistic regression cost
h = sigmoid(X @ W)
J = -(1/m) * (y @ np.log(h) + (1-y) @ np.log(1-h))

# Gradient (both linear and logistic)
grad = (1/m) * X.T @ (predictions - y)
W -= lr * grad

# One-hot
Y = np.zeros((m, K)); Y[np.arange(m), y] = 1

# Neural network forward
a0 = np.hstack([np.ones((m,1)), X])   # add bias
a1 = sigmoid(a0 @ W1)
a1b = np.hstack([np.ones((m,1)), a1]) # add bias
a2 = sigmoid(a1b @ W2)

# K-means assign
dists = np.array([(X - c)**2).sum(axis=1) for c in centroids]).T  # (m, K)
assignments = dists.argmin(axis=1)

# PCA
Sigma = (1/m) * X_norm.T @ X_norm
eigvals, eigvecs = np.linalg.eig(Sigma)
order = np.argsort(eigvals)[::-1]
U = eigvecs[:, order][:, :k]    # top k components
X_reduced = X_norm @ U
```

**Next:** [04_matplotlib_quick.ipynb](04_matplotlib_quick.ipynb) — plots for visualizing these algorithms.